[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LotfiDjebbar/atelier-transformers/blob/main/Transformer_Grand_Prix_Serveur.ipynb)

# 🏎️ Transformer Grand Prix — Serveur de course (animateur)

Ce notebook est **réservé à l'animateur**. Il lance un mini site web :

| Page | Qui l'ouvre | Ce qu'on y voit |
|---|---|---|
| **Dashboard** | Le projecteur | Le circuit, les voitures qui avancent, le chrono, le classement en direct, les feux de départ, le podium |
| **Cockpit** (`/play`) | Chaque étudiant, sur son PC | Son secteur à résoudre, son chrono, sa position, et le **garage** : des outils qui font tourner le vrai modèle |

Le modèle tourne ici, dans Colab. Les étudiants n'ont rien à installer : un navigateur suffit. Ils peuvent aussi scanner le QR code affiché au tableau.

👉 **Exécutez toutes les cellules dans l'ordre** (Exécution → Tout exécuter). Ça prend environ 2 minutes.

### 1. Installation

On installe les bibliothèques du modèle, **Flask** (le serveur web), **segno** (le QR code) et **cloudflared**, qui crée gratuitement un lien public vers ce Colab, sans compte.

In [ ]:
!pip install -q transformers sentencepiece sacremoses flask segno
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

### 2. Le moteur : MarianMT

Le même modèle que dans le notebook de cours, avec les mêmes fonctions (tokens, traduction, carte de cross-attention, décodage pas à pas). Le verrou empêche deux étudiants d'utiliser le modèle exactement en même temps.

Si un GPU est disponible (Exécution → Modifier le type d'exécution → GPU), il est utilisé automatiquement, mais le CPU suffit.

In [ ]:
import threading
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

torch.set_grad_enabled(False)

nom_modele = "Helsinki-NLP/opus-mt-fr-en"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
model = AutoModelForSeq2SeqLM.from_pretrained(nom_modele, attn_implementation="eager")
appareil = "cuda" if torch.cuda.is_available() else "cpu"
model.to(appareil)
verrou_modele = threading.Lock()


def encoder(texte):
    return tokenizer(texte, return_tensors="pt", truncation=True, max_length=64).to(appareil)


def traduire(texte, strategie="Beam search", temperature=1.0):
    if strategie == "Greedy":
        options = dict(num_beams=1, do_sample=False)
    elif strategie == "Sampling":
        options = dict(num_beams=1, do_sample=True, temperature=float(temperature), top_k=0)
    else:
        options = dict(num_beams=5, do_sample=False)
    with verrou_modele:
        sortie = model.generate(**encoder(texte), max_new_tokens=60, **options)
    return tokenizer.decode(sortie[0], skip_special_tokens=True)


def tokens(texte):
    ids = tokenizer(texte, truncation=True, max_length=64).input_ids
    return {"tokens": tokenizer.convert_ids_to_tokens(ids), "ids": ids}


def decoder_ids(ids):
    return tokenizer.decode(ids, skip_special_tokens=True)


def carte(texte):
    with verrou_modele:
        entree = encoder(texte)
        traduction = model.generate(**entree, num_beams=1, do_sample=False, max_new_tokens=60)
        res = model(**entree, decoder_input_ids=traduction[:, :-1], output_attentions=True)
    poids = torch.stack(res.cross_attentions).mean(dim=(0, 2))[0, :, :-1]
    poids = poids / poids.sum(-1, keepdim=True)
    return {
        "source": tokenizer.convert_ids_to_tokens(entree.input_ids[0])[:-1],
        "cible": tokenizer.convert_ids_to_tokens(traduction[0, 1:]),
        "poids": poids.cpu().tolist(),
    }


def pas_a_pas(texte):
    etapes = []
    with verrou_modele:
        entree = encoder(texte)
        memoire = model.get_encoder()(**entree)
        ecrit = torch.tensor([[model.config.decoder_start_token_id]], device=appareil)
        for _ in range(60):
            logits = model(encoder_outputs=memoire, attention_mask=entree.attention_mask, decoder_input_ids=ecrit).logits[0, -1]
            logits[tokenizer.pad_token_id] = -float("inf")
            top = logits.softmax(-1).topk(3)
            etapes.append([[tokenizer.convert_ids_to_tokens(int(i)), round(float(p), 3)] for p, i in zip(top.values, top.indices)])
            ecrit = torch.cat([ecrit, top.indices[:1].view(1, 1)], dim=1)
            if int(top.indices[0]) == tokenizer.eos_token_id:
                break
    return {"etapes": etapes, "traduction": tokenizer.decode(ecrit[0], skip_special_tokens=True)}

### 3. Les règles et les secteurs

Exécutez sans ouvrir 😉 Cette cellule calcule les réponses à partir du modèle et gère les chronos, les pénalités et les secteurs violets.

| Secteur | Notion | Ce qu'il faut faire | Garage |
|---|---|---|---|
| 1. Le compte-tours | Encoder vs decoder | Compter les passages de l'encoder et du decoder pour 3 phrases (piège : l'encoder tourne une fois **par phrase**) | Moteur pas à pas |
| 2. La chicane des masques | Masked self-attention + cross-attention | Compter les poids non masqués dans une tête (triangle + rectangle) | Tokenizer (pour compter les tokens source) |
| 3. Le duel Q·K | Query / Key | Calculer 4 produits scalaires à la main | 🧠 Aucun : bloc-notes |
| 4. Le virage de la température | Softmax et sampling | Calculer exp((3 − 2) / 0,5) ≈ 7 | 🧠 Aucun : bloc-notes |
| 5. Le muret du beam search | Greedy vs beam search | Trouver la phrase que le beam search préfère au greedy (A cat, 0,36) | 🧠 Aucun : bloc-notes |
| 6. Drapeau à damier | Pipeline complet | Retrouver une phrase française (une expression de pilote) qui donne exactement une traduction anglaise | Traducteur |

Les noms de pilotes sont **uniques** : majuscules, accents et espaces ne comptent pas (« Lotfi », « lotfi » et « LÔTFI » sont le même nom).

In [ ]:
#@title ⚙️ Règles du Grand Prix
import time, random, uuid, collections, re, math, unicodedata

code_admin = str(random.randint(1000, 9999))
PENALITE_ERREUR = 15
PENALITE_STAND = 30


def normaliser(x):
    return "".join(c for c in str(x).lower() if c.isalnum() or c == ",")


def cle_nom(nom):
    sans_accents = unicodedata.normalize("NFKD", nom).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", sans_accents.lower())


def nombre(texte):
    trouve = re.search(r"\d+(?:[.,]\d+)?", str(texte))
    return float(trouve.group(0).replace(",", ".")) if trouve else None


messages_radio = ["La voiture démarre.", "Le pilote freine.", "Le moteur chauffe."]
reponse_compte_tours = f"{len(messages_radio)},{sum(len(pas_a_pas(m)['etapes']) for m in messages_radio)}"

phrase_masque = "La voiture rouge gagne"
tokens_masque = len(tokens(phrase_masque)["ids"])
reponse_masque = str(3 * 4 // 2 + 3 * tokens_masque)

phrase_drapeau = "Il appuie sur le champignon dans la dernière ligne droite."
cible_drapeau = traduire(phrase_drapeau)


def verifier_compte_tours(r):
    return normaliser(r) == reponse_compte_tours, None


def verifier_masque(r):
    return normaliser(r) == reponse_masque, None


def verifier_qk(r):
    n = nombre(r)
    mots = re.sub(r"[^a-z]", "", str(r).lower())
    return mots == "chat" and n == 4, None


def verifier_temperature(r):
    n = nombre(r)
    return n is not None and abs(n - math.exp(2)) < 0.6, None


def verifier_beam(r):
    n = nombre(r)
    mots = re.sub(r"[^a-z]", "", str(r).lower())
    return mots == "acat" and n is not None and (abs(n - 0.36) < 0.006 or abs(n - 36) < 0.6), None


def verifier_drapeau(r):
    mots_cible = set(re.findall(r"[a-z']+", cible_drapeau.lower()))
    mots_reponse = re.findall(r"[a-z']+", str(r).lower())
    if mots_reponse and sum(m in mots_cible for m in mots_reponse) / len(mots_reponse) > 0.6:
        return False, "En français, pilote ! Le moteur traduit du français vers l'anglais 😉"
    traduction = traduire(r)
    return normaliser(traduction) == normaliser(cible_drapeau), f"Le moteur a traduit : « {traduction} »"


GRILLE_BEAM = """<table class="arbre">
<tr><th>Étape 1</th><th>Étape 2</th></tr>
<tr><td>« The » 0,6</td><td>après « The » : « cat » 0,5 · « dog » 0,4 · « cow » 0,1</td></tr>
<tr><td>« A » 0,4</td><td>après « A » : « cat » 0,9 · « dog » 0,1</td></tr></table>"""

SECTEURS = [
    {"titre": "Le compte-tours", "notion": "Encoder vs decoder", "type": "texte", "outil": "pas",
     "enonce": "L'écurie fait traduire <b>3 messages radio</b>, l'un après l'autre :<br>"
               + "<br>".join(f"• <b>{m}</b>" for m in messages_radio)
               + "<br><br>Au total, combien de passages de l'<b>encoder</b> et combien de passages du <b>decoder</b> ?<br>Format : <b>encoder,decoder</b> (ex. 5,40)",
     "indice": "L'encoder tourne une fois <b>par phrase source</b>. Le decoder tourne une fois par token écrit, &lt;/s&gt; compris. Le moteur pas à pas vous montre les tours du decoder pour une phrase.",
     "verifier": verifier_compte_tours},
    {"titre": "La chicane des masques", "notion": "Masked self-attention et cross-attention", "type": "texte", "outil": "tokens",
     "enonce": f"Le moteur traduit <b>{phrase_masque}</b>. Le decoder a déjà en entrée <b>3 tokens</b> : « &lt;s&gt; The red ».<br><br>"
               "Dans <b>une seule tête</b>, combien de poids d'attention <b>non masqués</b> le decoder calcule-t-il au total "
               "(masked self-attention + cross-attention) ?<br><br>Attention : l'encoder voit la phrase source après le tokenizer, &lt;/s&gt; compris.",
     "indice": "Masked self-attention : 3 tokens, chacun ne voit que lui-même et le passé (grille triangulaire). Cross-attention : chacun des 3 tokens voit <b>tous</b> les tokens source. Comptez les tokens source avec le tokenizer.",
     "verifier": verifier_masque},
    {"titre": "Le duel Q·K", "notion": "Query, Key, cross-attention", "type": "texte", "outil": "cerveau",
     "enonce": "Le decoder va écrire « cat ». Sa <b>query</b> vaut <b>q = [1, −1, 2]</b>.<br>Les <b>keys</b> de l'encoder pour « Le chat noir dort » :<br>"
               "• Le : [3, 1, 0]<br>• chat : [0, −2, 1]<br>• noir : [1, 0, 1]<br>• dort : [1, 2, 1]<br><br>"
               "Quel mot obtient le plus grand score <b>q·k</b> (avant softmax), et quel est ce score ?<br>Format : <b>mot,score</b> (ex. dort,7)",
     "indice": "Produit scalaire : q·k = q1×k1 + q2×k2 + q3×k3. Calculez-le pour les 4 mots.",
     "verifier": verifier_qk},
    {"titre": "Le virage de la température", "notion": "Softmax et sampling", "type": "texte", "outil": "cerveau",
     "enonce": "Au 3e mot, les logits des deux meilleurs candidats sont : <b>« sleeps » = 3</b> et <b>« is » = 2</b>.<br>"
               "Avec le sampling à la température <b>T = 0,5</b>, combien de fois « sleeps » est-il plus probable que « is » ?<br>"
               "Arrondi à l'entier. (e ≈ 2,718)",
     "indice": "Avec la température, p<sub>i</sub> ∝ exp(logit<sub>i</sub> / T). Le rapport des deux probabilités vaut exp((3 − 2) / T).",
     "verifier": verifier_temperature},
    {"titre": "Le muret du beam search", "notion": "Greedy vs beam search", "type": "texte", "outil": "cerveau",
     "enonce": f"Le moteur écrit une traduction de 2 mots. Voici ses probabilités :<br>{GRILLE_BEAM}<br>"
               "Le <b>greedy</b> choisit « The » puis « cat ». Quelle phrase le <b>beam search (k = 2)</b> choisit-il, et quelle est sa probabilité ?<br>"
               "Format : <b>phrase,probabilité</b> (ex. The dog,0.24)",
     "indice": "La probabilité d'une phrase = produit des probabilités de ses mots. Le beam garde les 2 meilleurs débuts, puis compare les phrases complètes.",
     "verifier": verifier_beam},
    {"titre": "Drapeau à damier", "notion": "Pipeline encoder-decoder complet", "type": "texte", "outil": "traduire",
     "enonce": f"Dernier tour ! Un commentateur a crié une phrase en français. Le moteur l'a traduite par :<br><b>« {cible_drapeau} »</b><br><br>"
               "Retrouvez une phrase <b>en français</b> que le moteur traduit <b>exactement</b> ainsi (beam search). Testez vos idées au garage.",
     "indice": "C'est une expression de pilote pressé : quand on accélère à fond, on « appuie sur le … ». Regardez aussi le temps des verbes.",
     "verifier": verifier_drapeau},
]

COULEURS = ["#ff2d55", "#ff9f0a", "#ffd60a", "#30d158", "#64d2ff", "#0a84ff", "#bf5af2", "#ff6ac1", "#ffffff", "#a2845e"]

RATES = [
    "Sortie de piste ! +15 s",
    "Limites de piste dépassées. +15 s",
    "Le moteur cale… +15 s",
    "Tête-à-queue dans le bac à graviers. +15 s",
    "Drapeau jaune : mauvaise réponse. +15 s",
]

course = {"etat": "attente", "debut": None, "fin": None, "pause": None, "cumul_pause": 0.0}
pilotes = {}
evenements = collections.deque(maxlen=14)
premiers = {}
verrou_jeu = threading.Lock()


def maintenant():
    return time.time()


def chrono():
    if course["debut"] is None or maintenant() < course["debut"]:
        return 0
    reference = course["fin"] or course["pause"] or maintenant()
    return max(0, reference - course["debut"] - course["cumul_pause"])


def en_course():
    return course["etat"] == "course" and course["pause"] is None and maintenant() >= course["debut"]


def evenement(texte, genre="info"):
    evenements.appendleft({"t": round(chrono(), 1), "texte": texte, "genre": genre})


def total(p):
    return (p["temps"][-1] if p["temps"] else 0) + p["penalite"]


def classement():
    return sorted(pilotes.values(), key=lambda p: (-len(p["temps"]), total(p), p["numero"]))


def position(pid):
    return [p["id"] for p in classement()].index(pid) + 1


def remettre_a_zero(p):
    p.update(temps=[], penalite=0, erreurs=0, stands=set(), fini=False)


def rejoindre(nom, couleur):
    nom = re.sub(r"\s+", " ", (nom or "").strip())[:16]
    if len(cle_nom(nom)) < 2:
        return None, "Choisissez un nom d'au moins 2 lettres ou chiffres."
    if couleur not in COULEURS:
        couleur = random.choice(COULEURS)
    with verrou_jeu:
        if any(cle_nom(p["nom"]) == cle_nom(nom) for p in pilotes.values()):
            return None, f"Le nom « {nom} » est déjà pris. Choisissez-en un autre."
        pid = uuid.uuid4().hex[:12]
        numero = max([p["numero"] for p in pilotes.values()], default=0) + 1
        pilotes[pid] = {"id": pid, "nom": nom, "couleur": couleur, "numero": numero}
        remettre_a_zero(pilotes[pid])
        evenement(f"{nom} entre sur la grille", "grille")
    return pid, None


def repondre(pid, reponse):
    p = pilotes[pid]
    n = len(p["temps"])
    if course["pause"] is not None:
        return {"ok": False, "message": "🚩 Drapeau rouge : la course est suspendue."}
    if not en_course() or p["fini"]:
        return {"ok": False, "message": "La course n'est pas lancée."}
    ok, retour = SECTEURS[n]["verifier"](str(reponse)[:200])
    with verrou_jeu:
        if pid not in pilotes or len(p["temps"]) != n:
            return {"ok": True, "message": "Déjà validé."}
        if not ok:
            p["penalite"] += PENALITE_ERREUR
            p["erreurs"] += 1
            return {"ok": False, "message": random.choice(RATES), "retour": retour}
        p["temps"].append(round(chrono(), 1))
        violet = n not in premiers
        if violet:
            premiers[n] = pid
            evenement(f"🟣 SECTEUR VIOLET — {p['nom']} est le premier à boucler « {SECTEURS[n]['titre']} »", "violet")
        else:
            evenement(f"{p['nom']} boucle le secteur {n + 1}", "secteur")
        if len(p["temps"]) == len(SECTEURS):
            p["fini"] = True
            evenement(f"🏁 {p['nom']} franchit la ligne d'arrivée ! P{position(pid)}", "arrivee")
            if all(x["fini"] for x in pilotes.values()):
                course["fin"] = maintenant()
                course["etat"] = "fini"
        return {"ok": True, "violet": violet, "message": "Secteur bouclé !"}


def stand(pid):
    p = pilotes[pid]
    n = len(p["temps"])
    if not en_course() or p["fini"]:
        return {"indice": ""}
    with verrou_jeu:
        if n not in p["stands"]:
            p["stands"].add(n)
            p["penalite"] += PENALITE_STAND
            evenement(f"🔧 {p['nom']} passe au stand (secteur {n + 1})", "stand")
    return {"indice": SECTEURS[n]["indice"]}


def admin(action, numero=None):
    with verrou_jeu:
        if action == "depart":
            for p in pilotes.values():
                remettre_a_zero(p)
            premiers.clear()
            evenements.clear()
            course.update(etat="course", debut=maintenant() + 6, fin=None, pause=None, cumul_pause=0.0)
        elif action == "pause" and en_course():
            course["pause"] = maintenant()
            evenement("🚩 DRAPEAU ROUGE — course suspendue", "pause")
        elif action == "reprendre" and course["pause"] is not None:
            course["cumul_pause"] += maintenant() - course["pause"]
            course["pause"] = None
            evenement("🟢 Drapeau vert — la course reprend !", "reprise")
        elif action == "fin" and course["etat"] == "course":
            if course["pause"] is not None:
                course["cumul_pause"] += maintenant() - course["pause"]
                course["pause"] = None
            course.update(etat="fini", fin=maintenant())
        elif action == "reset":
            course.update(etat="attente", debut=None, fin=None, pause=None, cumul_pause=0.0)
            for p in pilotes.values():
                remettre_a_zero(p)
            premiers.clear()
            evenements.clear()
        elif action == "vider":
            course.update(etat="attente", debut=None, fin=None, pause=None, cumul_pause=0.0)
            pilotes.clear()
            premiers.clear()
            evenements.clear()
        elif action == "retirer":
            for pid, p in list(pilotes.items()):
                if p["numero"] == numero:
                    del pilotes[pid]
                    for k in [k for k, v in premiers.items() if v == pid]:
                        del premiers[k]
                    evenement(f"{p['nom']} est retiré de la course", "retrait")
            if course["etat"] == "course" and pilotes and all(x["fini"] for x in pilotes.values()):
                course.update(etat="fini", fin=maintenant())


def vue_pilote(p):
    return {"numero": p["numero"], "nom": p["nom"], "couleur": p["couleur"], "secteurs": len(p["temps"]),
            "temps": p["temps"], "penalite": p["penalite"], "total": round(total(p), 1), "fini": p["fini"],
            "violets": [n for n, i in premiers.items() if i == p["id"]]}


def etat_commun():
    return {"maintenant": maintenant(), "etat": course["etat"], "debut": course["debut"], "chrono": round(chrono(), 1),
            "fin": course["fin"], "pause": course["pause"], "cumul_pause": course["cumul_pause"],
            "nb_secteurs": len(SECTEURS), "titres": [s["titre"] for s in SECTEURS]}


def tableau():
    d = etat_commun()
    d["pilotes"] = [vue_pilote(p) for p in classement()]
    d["evenements"] = list(evenements)
    return d


def etat_pilote(pid):
    d = etat_commun()
    p = pilotes.get(pid)
    if p is None:
        d["inconnu"] = True
        return d
    d["moi"] = vue_pilote(p)
    d["moi"]["position"] = position(pid)
    d["moi"]["nb_pilotes"] = len(pilotes)
    n = len(p["temps"])
    if course["etat"] == "course" and maintenant() >= course["debut"] and not p["fini"]:
        s = SECTEURS[n]
        d["secteur"] = {"numero": n + 1, "titre": s["titre"], "notion": s["notion"], "enonce": s["enonce"],
                        "type": s["type"], "options": s.get("options", []), "outil": s["outil"],
                        "indice": s["indice"] if n in p["stands"] else ""}
    return d

### 4. Les pages web

Le dashboard du projecteur et le cockpit des pilotes.

In [ ]:
#@title 🖥️ Pages web
PAGE_DASHBOARD = r'''<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Transformer Grand Prix — Direction de course</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@600;800;900&family=Titillium+Web:wght@400;600;700&display=swap" rel="stylesheet">
<style>
:root{--bg:#07090c;--panel:#10141a;--line:#1f2630;--txt:#e9eef5;--mut:#7d8a99;--red:#ff2d2d;--violet:#b04dff;--green:#2fe07a;--gold:#ffc83d}
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%;background:var(--bg);color:var(--txt);font-family:'Titillium Web','Segoe UI',system-ui,sans-serif;overflow:hidden}
body{display:grid;grid-template-rows:auto minmax(0,1fr) auto;grid-template-columns:minmax(0,1fr);height:100vh;width:100vw}
header{display:flex;align-items:center;justify-content:space-between;padding:14px 28px;background:linear-gradient(90deg,#14060a,#07090c 55%);border-bottom:3px solid var(--red)}
.logo{display:flex;align-items:center;gap:16px}
.damier{width:46px;height:46px;background:conic-gradient(#fff 0 25%,#000 0 50%,#fff 0 75%,#000 0) 0 0/15.3px 15.3px;border-radius:6px;transform:skew(-8deg)}
.logo h1{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-size:30px;letter-spacing:2px;font-style:italic}
.logo h1 span{color:var(--red)}
.logo small{display:block;font-size:13px;color:var(--mut);letter-spacing:4px;text-transform:uppercase}
.statut{display:flex;align-items:center;gap:22px}
.badge{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:14px;padding:7px 14px;border-radius:6px;letter-spacing:2px;background:#1b222c;color:var(--mut)}
.badge.live{background:var(--red);color:#fff;animation:pulse 1.4s infinite}
.badge.fini{background:#fff;color:#000}
@keyframes pulse{50%{opacity:.6}}
.horloge{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:800;font-size:52px;font-variant-numeric:tabular-nums;letter-spacing:2px;min-width:250px;text-align:right}
main{display:grid;grid-template-columns:minmax(0,1.45fr) minmax(0,1fr);gap:18px;padding:18px 24px;min-height:0}
.carte{background:var(--panel);border:1px solid var(--line);border-radius:14px;position:relative;overflow:hidden;min-height:0}
.titre-bloc{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:13px;letter-spacing:3px;color:var(--mut);padding:14px 18px 0;text-transform:uppercase}
#circuit{width:100%;height:calc(100% - 34px)}
.tour{display:flex;flex-direction:column;min-height:0}
.lignes{flex:1;overflow:hidden;padding:10px 12px 12px}
.ligne{display:grid;grid-template-columns:40px 34px minmax(0,1fr) auto 96px;align-items:center;gap:12px;height:46px;margin-bottom:6px;padding:0 12px;background:#151a22;border-radius:8px;transition:transform .5s;font-size:20px}
.ligne:nth-child(1){background:linear-gradient(90deg,#3a2c05,#151a22 60%)}
.pos{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:800;font-size:20px;text-align:center}
.barre{width:32px;height:32px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-family:Orbitron,'Arial Black',sans-serif;font-weight:900;font-size:15px;color:#000;border:2px solid #000}
.nom{font-weight:700;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.nom em{font-style:normal;color:var(--gold);font-size:15px;margin-left:6px}
.secs{display:flex;gap:4px}
.sec{width:15px;height:15px;border-radius:3px;background:#262e3a}
.sec.ok{background:var(--green)}
.sec.violet{background:var(--violet);box-shadow:0 0 8px var(--violet)}
.temps{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:16px;text-align:right;font-variant-numeric:tabular-nums}
.temps small{display:block;font-size:11px;color:#ff7b7b}
footer{min-width:0;overflow:hidden;display:flex;align-items:center;gap:16px;padding:10px 24px 14px;min-height:62px}
.radio{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:12px;letter-spacing:3px;color:#000;background:var(--gold);padding:6px 10px;border-radius:4px;white-space:nowrap}
#fil{flex:1;min-width:0;display:flex;gap:26px;overflow:hidden;white-space:nowrap;font-size:19px}
#fil .ev{color:var(--mut)}
#fil .ev:first-child{color:var(--txt);font-weight:700}
#fil .violet{color:#d9a6ff}
#fil .arrivee{color:var(--gold)}
.voile{position:fixed;inset:0;background:rgba(4,5,8,.93);display:none;align-items:center;justify-content:center;z-index:10;flex-direction:column}
.voile.on{display:flex}
#feux .rangee{display:flex;gap:26px;padding:28px 34px;background:#111;border-radius:18px;border:3px solid #222}
.feu{width:120px;display:flex;flex-direction:column;gap:14px;align-items:center;background:#000;padding:14px 0;border-radius:14px}
.lampe{width:84px;height:84px;border-radius:50%;background:#2a0b0b;transition:background .1s,box-shadow .1s}
.feu.on .lampe{background:#ff1a1a;box-shadow:0 0 40px #ff1a1a,0 0 90px #ff1a1a80}
#go{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-size:150px;color:var(--green);margin-top:30px;height:180px;font-style:italic;text-shadow:0 0 40px #2fe07a80}
#grille .boite{display:grid;grid-template-columns:auto 1fr;gap:50px;align-items:center;max-width:1300px;padding:20px}
#grille h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:54px;font-style:italic;font-weight:900;line-height:1.05}
#grille h2 span{color:var(--red)}
#grille p{font-size:24px;color:var(--mut);margin:16px 0 8px}
#lien{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:34px;color:var(--gold);word-break:break-all}
#qr{display:block;width:340px;height:340px;background:#fff;border-radius:16px;padding:12px;object-fit:contain}
#inscrits{display:flex;flex-wrap:wrap;gap:10px;margin-top:22px;max-height:250px;overflow:hidden}
.puce{display:flex;align-items:center;gap:8px;background:#161c25;padding:6px 14px 6px 6px;border-radius:30px;font-size:20px;font-weight:600;animation:entree .4s}
.puce i{width:28px;height:28px;border-radius:50%;display:inline-flex;align-items:center;justify-content:center;font-style:normal;font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:13px;color:#000;font-weight:800}
@keyframes entree{from{transform:scale(.3);opacity:0}}
#podium h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:64px;font-style:italic;font-weight:900;margin-bottom:10px;background:conic-gradient(#fff 0 25%,#000 0 50%,#fff 0 75%,#000 0) 0 0/40px 40px;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-stroke:2px #fff}
#podium .marches{display:flex;align-items:flex-end;gap:22px;margin-top:40px}
.marche{width:270px;text-align:center}
.marche .qui{font-size:34px;font-weight:700;margin-bottom:10px}
.marche .qui small{display:block;font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:18px;color:var(--mut)}
.marche .bloc{border-radius:12px 12px 0 0;display:flex;align-items:flex-start;justify-content:center;padding-top:16px;font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-size:70px;color:#000}
#podium .sous{font-size:22px;color:var(--mut);margin-top:26px}
#admin{position:fixed;right:14px;bottom:76px;display:none;gap:8px;z-index:20}
#admin.on{display:flex}
#admin button{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:13px;letter-spacing:1px;border:0;border-radius:8px;padding:10px 14px;cursor:pointer;background:#232b36;color:#fff}
#admin button.vert{background:var(--green);color:#000}
#admin button.rouge{background:var(--red)}
#admin button.orange{background:#ff8a00;color:#000}
.retirer{display:none;border:0;background:#3a1418;color:#ff8a8a;border-radius:6px;width:26px;height:26px;cursor:pointer;font-weight:900;margin-left:8px;font-size:14px}
body.admin .retirer{display:inline-block}
.puce .retirer{width:22px;height:22px;margin-left:4px}
#drapeau{background:repeating-linear-gradient(45deg,#7a0000 0 60px,#5a0000 60px 120px)}
#drapeau .flag{font-size:150px;line-height:1}
#drapeau h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-style:italic;font-size:86px;margin-top:10px}
#drapeau p{font-size:30px;opacity:.85;margin-top:8px}
.badge.pause{background:#c00000;color:#fff}
.vide{color:var(--mut);font-size:20px;padding:30px;text-align:center}
</style>
</head>
<body>
<header>
  <div class="logo"><div class="damier"></div><div><h1>TRANSFORMER <span>GRAND PRIX</span></h1><small>Encoder · Decoder · Cross-attention</small></div></div>
  <div class="statut"><div class="badge" id="badge">GRILLE</div><div class="horloge" id="horloge">00:00.0</div></div>
</header>
<main>
  <section class="carte"><div class="titre-bloc">Circuit — 6 secteurs</div><svg id="circuit" viewBox="0 0 1000 560" preserveAspectRatio="xMidYMid meet"></svg></section>
  <section class="carte tour"><div class="titre-bloc">Classement en direct</div><div class="lignes" id="lignes"></div></section>
</main>
<footer><div class="radio">RADIO</div><div id="fil"></div></footer>

<div class="voile" id="grille"><div class="boite">
  <div><img id="qr" alt="QR code"></div>
  <div><h2>REJOIGNEZ<br>LA <span>GRILLE</span></h2><p>Ouvrez ce lien sur votre PC, choisissez votre pseudo et votre couleur :</p><div id="lien"></div><div id="inscrits"></div></div>
</div></div>
<div class="voile" id="feux"><div class="rangee" id="rangee"></div><div id="go"></div></div>
<div class="voile" id="drapeau"><div class="flag">🚩</div><h2>DRAPEAU ROUGE</h2><p>Course suspendue — chrono arrêté</p></div>
<div class="voile" id="podium"><h2>ARRIVÉE</h2><div class="marches" id="marches"></div><div class="sous">Cliquez pour fermer</div></div>
<div id="admin"><button class="vert" data-a="depart">Départ</button><button class="orange" data-a="pause" id="btn-pause">Pause</button><button data-a="fin">Fin de course</button><button data-a="reset">Reset</button><button class="rouge" data-a="vider">Vider la grille</button></div>

<script>
const code = new URLSearchParams(location.search).get("admin");
const SVGNS = "http://www.w3.org/2000/svg";
const esc = s => String(s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const $ = id => document.getElementById(id);
let decalage = 0, dernier = null, podiumFerme = false, audio = null, dernierBip = -1;

const D = "M 150 470 L 610 470 C 760 470 830 430 870 345 C 905 265 900 165 825 115 C 760 72 690 92 650 145 C 612 195 560 205 510 172 C 452 135 425 72 330 72 C 222 72 152 122 122 205 C 98 272 112 330 92 380 C 76 425 92 470 150 470 Z";
const svg = $("circuit");
function el(nom, attrs, parent){const e=document.createElementNS(SVGNS,nom);for(const k in attrs)e.setAttribute(k,attrs[k]);(parent||svg).appendChild(e);return e;}
el("path",{d:D,fill:"none",stroke:"#e8e8e8","stroke-width":64,"stroke-linejoin":"round"});
el("path",{d:D,fill:"none",stroke:"#e01e1e","stroke-width":64,"stroke-dasharray":"16 16","stroke-linejoin":"round"});
el("path",{d:D,fill:"none",stroke:"#2a2f37","stroke-width":52,"stroke-linejoin":"round"});
el("path",{d:D,fill:"none",stroke:"#ffffff30","stroke-width":2,"stroke-dasharray":"18 22"});
const piste = el("path",{d:D,fill:"none",stroke:"none"});
const L = piste.getTotalLength();
const d0 = piste.getPointAtLength(0), d1 = piste.getPointAtLength(1);
const angle = Math.atan2(d1.y-d0.y, d1.x-d0.x)*180/Math.PI;
const ligne = el("g",{transform:`translate(${d0.x},${d0.y}) rotate(${angle})`});
for(let i=0;i<10;i++)for(let j=0;j<2;j++)el("rect",{x:j*7-7,y:-26+i*5.2,width:7,height:5.2,fill:(i+j)%2?"#000":"#fff"},ligne);
const NB = 6;
for(let k=1;k<NB;k++){
  const p = piste.getPointAtLength(L*k/NB), q = piste.getPointAtLength(L*k/NB+1);
  const a = Math.atan2(q.y-p.y,q.x-p.x);
  const nx = -Math.sin(a), ny = Math.cos(a);
  el("line",{x1:p.x-nx*26,y1:p.y-ny*26,x2:p.x+nx*26,y2:p.y+ny*26,stroke:"#ffc83d","stroke-width":3});
  const t = el("text",{x:p.x+nx*52,y:p.y+ny*52+6,"text-anchor":"middle",fill:"#ffc83d","font-family":"Orbitron, Arial Black, sans-serif","font-size":18,"font-weight":800});
  t.textContent = "S"+k;
}
const tArr = el("text",{x:d0.x,y:d0.y+62,"text-anchor":"middle",fill:"#fff","font-family":"Orbitron, Arial Black, sans-serif","font-size":15,"font-weight":800});
tArr.textContent = "DÉPART / ARRIVÉE";
const calqueVoitures = el("g",{});
const voitures = {};

function voiture(p){
  let v = voitures[p.numero];
  if(!v){
    const g = el("g",{},calqueVoitures);
    el("circle",{r:17,fill:p.couleur,stroke:"#000","stroke-width":3},g);
    const t = el("text",{y:6,"text-anchor":"middle","font-family":"Orbitron, Arial Black, sans-serif","font-size":15,"font-weight":900,fill:"#000"},g);
    t.textContent = p.numero;
    v = voitures[p.numero] = {g, l:-60, cible:-60};
  }
  return v;
}

function placerVoitures(pilotes){
  const groupes = {};
  const vus = new Set();
  pilotes.forEach(p => {
    const v = voiture(p);
    vus.add(String(p.numero));
    const rang = groupes[p.secteurs] = (groupes[p.secteurs]||0);
    groupes[p.secteurs]++;
    v.cible = p.secteurs===NB ? L + 34 + rang*34 : L*p.secteurs/NB - 26 - rang*34;
  });
  for(const k in voitures) if(!vus.has(k)){voitures[k].g.remove();delete voitures[k];}
}

function animer(){
  for(const k in voitures){
    const v = voitures[k];
    const diff = v.cible - v.l;
    v.l += Math.abs(diff) < 1 ? diff : diff*0.045 + Math.sign(diff)*1.2;
    const p = piste.getPointAtLength(((v.l % L)+L)%L);
    v.g.setAttribute("transform",`translate(${p.x},${p.y})`);
  }
  requestAnimationFrame(animer);
}
requestAnimationFrame(animer);

function fmt(s){s=Math.max(0,s);const m=Math.floor(s/60), r=s-m*60;return String(m).padStart(2,"0")+":"+r.toFixed(1).padStart(4,"0");}

for(let i=0;i<5;i++){const f=document.createElement("div");f.className="feu";f.innerHTML='<div class="lampe"></div><div class="lampe"></div>';$("rangee").appendChild(f);}

function bip(freq,duree){
  if(!audio) return;
  const o = audio.createOscillator(), g = audio.createGain();
  o.type="square"; o.frequency.value=freq; g.gain.value=.08;
  o.connect(g); g.connect(audio.destination); o.start(); o.stop(audio.currentTime+duree);
}

function rendre(d){
  dernier = d;
  const lien = location.origin + "/play";
  if($("qr").dataset.lien !== lien){
    $("qr").dataset.lien = lien;
    $("lien").textContent = lien.replace(/^https?:\/\//,"");
    $("qr").src = "/qr.svg?u="+encodeURIComponent(lien);
  }
  const live = d.etat==="course" && d.maintenant>=d.debut && !d.pause;
  $("badge").className = "badge"+(d.pause?" pause":live?" live":d.etat==="fini"?" fini":"");
  $("badge").textContent = d.pause?"DRAPEAU ROUGE":live?"EN COURSE":d.etat==="fini"?"DRAPEAU À DAMIER":d.etat==="course"?"DÉPART":"GRILLE";
  $("drapeau").classList.toggle("on", !!d.pause);
  $("btn-pause").textContent = d.pause ? "Reprendre" : "Pause";
  $("btn-pause").dataset.a = d.pause ? "reprendre" : "pause";
  $("grille").classList.toggle("on", d.etat==="attente");
  const puces = d.pilotes.map(p=>`<div class="puce"><i style="background:${p.couleur}">${p.numero}</i>${esc(p.nom)}<button class="retirer" data-n="${p.numero}" data-nom="${esc(p.nom)}" title="Retirer">✕</button></div>`).join("") || '<div class="vide">En attente des pilotes…</div>';
  if($("inscrits").dataset.html !== puces){ $("inscrits").dataset.html = puces; $("inscrits").innerHTML = puces; }
  const tete = d.pilotes[0];
  $("lignes").innerHTML = d.pilotes.length ? d.pilotes.map((p,i)=>{
    const secs = Array.from({length:d.nb_secteurs},(_,k)=>`<div class="sec ${k<p.secteurs?(p.violets.includes(k)?"violet":"ok"):""}"></div>`).join("");
    let t;
    if(p.fini) t = fmt(p.total);
    else if(p.secteurs===0) t = "—";
    else if(i>0 && tete && tete.secteurs===p.secteurs) t = "+"+(p.total-tete.total).toFixed(1);
    else t = "S"+p.secteurs;
    return `<div class="ligne"><div class="pos">${i+1}</div><div class="barre" style="background:${p.couleur}">${p.numero}</div><div class="nom">${esc(p.nom)}${p.fini?"<em>🏁</em>":""}<button class="retirer" data-n="${p.numero}" data-nom="${esc(p.nom)}" title="Retirer">✕</button></div><div class="secs">${secs}</div><div class="temps">${t}${p.penalite?`<small>+${p.penalite}s pén.</small>`:""}</div></div>`;
  }).join("") : '<div class="vide">Aucun pilote pour l\'instant</div>';
  $("fil").innerHTML = d.evenements.map(e=>`<span class="ev ${e.genre}">${fmt(e.t)} · ${esc(e.texte)}</span>`).join("");
  placerVoitures(d.pilotes);
  const fin = d.etat==="fini" || (d.pilotes.length && d.pilotes.every(p=>p.fini));
  if(!fin) podiumFerme = false;
  $("podium").classList.toggle("on", fin && !podiumFerme);
  if(fin){
    const ordre=[1,0,2], hauteurs=[260,190,140], cols=["#ffc83d","#d7dde4","#cd8b52"];
    $("marches").innerHTML = ordre.filter(i=>d.pilotes[i]).map(i=>{const p=d.pilotes[i];return `<div class="marche"><div class="qui"><span style="color:${p.couleur}">●</span> ${esc(p.nom)}<small>${p.fini?fmt(p.total):"S"+p.secteurs}</small></div><div class="bloc" style="height:${hauteurs[i]}px;background:${cols[i]}">${i+1}</div></div>`;}).join("");
  }
}

function tick(){
  if(dernier){
    const d = dernier, now = Date.now()/1000 + decalage;
    let c = 0;
    if(d.debut && now>=d.debut && d.etat!=="attente") c = (d.fin || d.pause || now) - d.debut - (d.cumul_pause||0);
    $("horloge").textContent = fmt(c);
    const reste = d.debut ? d.debut - now : 99;
    const montrer = d.etat==="course" && reste > -1.6;
    $("feux").classList.toggle("on", montrer);
    if(montrer){
      const allumes = reste>0 ? Math.min(5, Math.max(0, Math.ceil(6-reste)-0)) : 0;
      document.querySelectorAll(".feu").forEach((f,i)=>f.classList.toggle("on", reste>0 && i<allumes && reste<5.99));
      $("go").textContent = reste<=0 ? "GO !" : "";
      const etape = reste>0 ? allumes : 6;
      if(etape!==dernierBip){dernierBip=etape; if(etape>=1&&etape<=5) bip(440,.25); if(etape===6) bip(880,.6);}
    } else dernierBip = -1;
  }
  requestAnimationFrame(tick);
}
requestAnimationFrame(tick);

async function charger(){
  try{
    const r = await fetch("/api/tableau");
    const d = await r.json();
    decalage = d.maintenant - Date.now()/1000;
    rendre(d);
  }catch(e){}
}
async function boucle(){ await charger(); setTimeout(boucle, 1000); }
boucle();

$("podium").onclick = () => {podiumFerme = true; $("podium").classList.remove("on");};
if(code){
  document.body.classList.add("admin");
  $("admin").classList.add("on");
  const retirer = async e => {
    const b = e.target.closest(".retirer"); if(!b) return;
    if(!confirm("Retirer « "+b.dataset.nom+" » de la course ?")) return;
    await fetch("/api/admin",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({code, action:"retirer", numero:+b.dataset.n})});
    charger();
  };
  $("lignes").addEventListener("click", retirer);
  $("inscrits").addEventListener("click", retirer);
  document.querySelectorAll("#admin button").forEach(b => b.onclick = async () => {
    if(b.dataset.a==="vider" && !confirm("Supprimer tous les pilotes ?")) return;
    if(!audio){ try{ audio = new (window.AudioContext||window.webkitAudioContext)(); }catch(e){} }
    await fetch("/api/admin",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({code, action:b.dataset.a})});
    charger();
  });
}
</script>
</body>
</html>
'''

PAGE_PILOTE = r'''<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Transformer Grand Prix — Cockpit</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@600;800;900&family=Titillium+Web:wght@400;600;700&display=swap" rel="stylesheet">
<style>
:root{--bg:#07090c;--panel:#10141a;--line:#1f2630;--txt:#e9eef5;--mut:#7d8a99;--red:#ff2d2d;--violet:#b04dff;--green:#2fe07a;--gold:#ffc83d}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--bg) radial-gradient(ellipse at top,#1a0a0e,transparent 60%);color:var(--txt);font-family:'Titillium Web','Segoe UI',system-ui,sans-serif;min-height:100vh;font-size:17px}
.ecran{display:none;max-width:1180px;margin:0 auto;padding:22px}
.ecran.on{display:block}
h1{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-style:italic;font-size:34px;letter-spacing:1px}
h1 span{color:var(--red)}
.sous{color:var(--mut);letter-spacing:3px;text-transform:uppercase;font-size:13px;margin-top:4px}
.panneau{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:22px;margin-top:20px}
label{display:block;font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:12px;letter-spacing:2px;color:var(--mut);margin:14px 0 8px;text-transform:uppercase}
input[type=text],textarea,select{width:100%;background:#0b0e13;border:1px solid #2a3340;color:var(--txt);border-radius:10px;padding:12px 14px;font:inherit;font-size:18px;outline:none}
input[type=text]:focus,textarea:focus{border-color:var(--gold)}
.couleurs{display:flex;flex-wrap:wrap;gap:10px}
.couleurs button{width:46px;height:46px;border-radius:50%;border:3px solid transparent;cursor:pointer}
.couleurs button.choisi{border-color:#fff;box-shadow:0 0 0 3px #000 inset}
.btn{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:800;letter-spacing:1px;border:0;border-radius:10px;padding:13px 22px;font-size:15px;cursor:pointer;background:var(--red);color:#fff;transition:transform .1s}
.btn:active{transform:scale(.97)}
.btn.gris{background:#232b36}
.btn.or{background:var(--gold);color:#000}
.btn:disabled{opacity:.5;cursor:wait}
.casque{display:flex;align-items:center;gap:16px;margin-top:26px}
.numero{width:62px;height:62px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-size:24px;color:#000;border:3px solid #000;box-shadow:0 0 0 2px #fff3}
.grille-msg{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:26px;font-weight:800;font-style:italic;margin-top:30px}
.feux{display:flex;gap:14px;margin-top:24px}
.feu{width:58px;height:58px;border-radius:50%;background:#2a0b0b}
.feu.on{background:#ff1a1a;box-shadow:0 0 25px #ff1a1a}
#go{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-size:70px;color:var(--green);font-style:italic;height:90px;margin-top:10px}
.barre-haut{display:flex;align-items:center;gap:18px;flex-wrap:wrap;background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:12px 18px}
.barre-haut .info{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:13px;color:var(--mut);letter-spacing:1px}
.barre-haut .info b{display:block;color:var(--txt);font-size:22px;letter-spacing:0;font-variant-numeric:tabular-nums}
.secs{display:flex;gap:5px;margin-left:auto}
.sec{width:34px;height:12px;border-radius:3px;background:#262e3a}
.sec.ok{background:var(--green)}
.sec.violet{background:var(--violet)}
.sec.ici{background:var(--gold);animation:cl 1s infinite}
@keyframes cl{50%{opacity:.4}}
.jeu{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1.15fr);gap:18px;margin-top:18px}
@media(max-width:900px){.jeu{grid-template-columns:minmax(0,1fr)}}
.etiquette{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:12px;letter-spacing:3px;color:var(--gold);text-transform:uppercase}
.secteur h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-style:italic;font-size:28px;margin:6px 0 4px}
.notion{display:inline-block;background:#1b222c;color:var(--mut);padding:4px 10px;border-radius:20px;font-size:14px}
.enonce{font-size:20px;line-height:1.55;margin:18px 0}
.enonce code{font-family:ui-monospace,Menlo,monospace;font-size:15px;background:#0b0e13;padding:6px 8px;border-radius:6px;display:inline-block;margin:6px 0;word-break:break-all}
.actions{display:flex;gap:10px;margin-top:14px;flex-wrap:wrap}
.qcm{display:grid;grid-template-columns:1fr 1fr;gap:10px}
.qcm button{background:#161c25;border:2px solid #2a3340;color:var(--txt);border-radius:10px;padding:16px;font:inherit;font-size:18px;font-weight:600;cursor:pointer}
.qcm button:hover{border-color:var(--gold)}
.retour{margin-top:16px;padding:14px 16px;border-radius:10px;font-size:18px;display:none}
.retour.on{display:block}
.retour.ko{background:#3a0d12;color:#ffb3b8;border:1px solid #6b1a22}
.retour.ok{background:#0d3a20;color:#a8ffcb;border:1px solid #1a6b3d}
.retour.violet{background:#2c0d3a;color:#e3b8ff;border:1px solid #6b1a8a}
.indice{margin-top:14px;padding:12px 14px;border-left:4px solid var(--gold);background:#1d1a0c;border-radius:6px;display:none}
.indice.on{display:block}
.garage h3{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:16px;letter-spacing:2px;margin:6px 0 4px}
.garage p{color:var(--mut);font-size:15px}
.ligne-outil{display:flex;gap:10px;margin-top:12px;flex-wrap:wrap}
.ligne-outil input[type=text]{flex:1;min-width:200px}
.res{margin-top:14px;overflow:auto;max-height:430px}
.jeton{display:inline-flex;flex-direction:column;align-items:center;background:#1b222c;border-radius:8px;padding:6px 10px;margin:4px;font-family:ui-monospace,Menlo,monospace}
.jeton small{color:var(--mut);font-size:12px}
table.hm{border-collapse:collapse;font-size:13px}
table.hm td,table.hm th{padding:6px 8px;text-align:center;font-family:ui-monospace,Menlo,monospace}
table.hm th{color:var(--mut);font-weight:400}
table.pas{width:100%;border-collapse:collapse;font-family:ui-monospace,Menlo,monospace;font-size:14px}
table.pas td,table.pas th{padding:6px 8px;border-bottom:1px solid #1f2630;text-align:left}
table.pas td:nth-child(2){color:var(--green);font-weight:700}
.trad{font-size:20px;background:#0b0e13;padding:14px;border-radius:10px;border:1px solid #2a3340}
.fin h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:44px;font-style:italic;font-weight:900;margin:20px 0}
.damier{height:26px;background:conic-gradient(#fff 0 25%,#000 0 50%,#fff 0 75%,#000 0) 0 0/26px 26px;border-radius:6px;margin:18px 0}
.arbre{border-collapse:collapse;margin:10px 0;font-size:17px}
.arbre td,.arbre th{border:1px solid #2a3340;padding:8px 12px;text-align:left}
.arbre th{color:var(--gold);font-family:Orbitron,'Arial Black',sans-serif;font-size:12px;letter-spacing:2px}
.erreur{color:#ff8a8a;margin-top:12px;min-height:22px}
#rouge{position:fixed;inset:0;background:repeating-linear-gradient(45deg,#7a0000 0 50px,#5a0000 50px 100px);display:none;align-items:center;justify-content:center;flex-direction:column;z-index:50;text-align:center}
#rouge.on{display:flex}
#rouge div{font-size:110px}
#rouge h2{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-weight:900;font-style:italic;font-size:54px}
#rouge p{font-size:22px;opacity:.85}
textarea.notes{min-height:220px;font-family:ui-monospace,Menlo,monospace;font-size:16px;margin-top:12px}
.gros{font-family:Orbitron,'Arial Black',Impact,sans-serif;font-size:56px;font-weight:900;color:var(--gold)}
</style>
</head>
<body>

<div class="ecran" id="e-inscription">
  <h1>TRANSFORMER <span>GRAND PRIX</span></h1>
  <div class="sous">Votre moteur : un Transformer encoder-decoder</div>
  <div class="panneau">
    <label>Nom de pilote</label>
    <input type="text" id="nom" maxlength="16" placeholder="ex. Yasmine, TurboTokens…">
    <label>Couleur de votre voiture</label>
    <div class="couleurs" id="couleurs"></div>
    <div class="actions"><button class="btn" id="rejoindre">Rejoindre la grille</button></div>
    <div class="erreur" id="j-err"></div>
  </div>
</div>

<div class="ecran" id="e-grille">
  <h1>TRANSFORMER <span>GRAND PRIX</span></h1>
  <div class="casque"><div class="numero" id="g-num"></div><div><div class="etiquette">Pilote</div><div style="font-size:26px;font-weight:700" id="g-nom"></div></div></div>
  <div class="grille-msg" id="g-msg">Vous êtes sur la grille. Regardez le tableau…</div>
  <div class="feux" id="g-feux"></div>
  <div id="go"></div>
  <div class="panneau">
    <div class="etiquette">Le règlement</div>
    <p class="enonce" style="font-size:17px">6 secteurs à boucler, chacun sur une notion du cours. Chaque secteur a son outil dans le <b>garage</b> : il fait tourner le vrai modèle.<br>
    ❌ Mauvaise réponse : <b>+15 s</b> · 🔧 Arrêt au stand (indice) : <b>+30 s</b> · 🟣 Premier à boucler un secteur : <b>secteur violet</b> au tableau.<br>
    Le classement : le plus de secteurs, puis le meilleur temps (pénalités comprises).</p>
  </div>
</div>

<div class="ecran" id="e-course">
  <div class="barre-haut">
    <div class="numero" id="c-num" style="width:44px;height:44px;font-size:17px"></div>
    <div class="info">PILOTE<b id="c-nom"></b></div>
    <div class="info">CHRONO<b id="c-chrono">00:00.0</b></div>
    <div class="info">POSITION<b id="c-pos">–</b></div>
    <div class="info">PÉNALITÉS<b id="c-pen">0 s</b></div>
    <div class="secs" id="c-secs"></div>
  </div>
  <div class="jeu">
    <div class="panneau secteur" style="margin-top:0">
      <div class="etiquette" id="s-num"></div>
      <h2 id="s-titre"></h2>
      <span class="notion" id="s-notion"></span>
      <div class="enonce" id="s-enonce"></div>
      <div id="s-saisie"></div>
      <div class="indice" id="s-indice"></div>
      <div class="retour" id="s-retour"></div>
    </div>
    <div class="panneau garage" style="margin-top:0">
      <div class="etiquette">🔧 Garage</div>
      <div id="garage"></div>
    </div>
  </div>
</div>

<div id="rouge"><div>🚩</div><h2>DRAPEAU ROUGE</h2><p>La course est suspendue par la direction de course.<br>Le chrono est arrêté. Gardez vos réponses en tête…</p></div>

<div class="ecran" id="e-fin">
  <h1>TRANSFORMER <span>GRAND PRIX</span></h1>
  <div class="damier"></div>
  <div class="fin"><h2>DRAPEAU À DAMIER !</h2></div>
  <div class="panneau">
    <div class="etiquette">Votre temps</div><div class="gros" id="f-temps"></div>
    <div class="etiquette" style="margin-top:16px">Position</div><div class="gros" id="f-pos"></div>
    <p class="enonce">Vous n'êtes plus de simples utilisateurs de Transformers : vous êtes des pilotes de Transformers. Regardez le classement au tableau !</p>
  </div>
</div>

<script>
const COULEURS = ["#ff2d55","#ff9f0a","#ffd60a","#30d158","#64d2ff","#0a84ff","#bf5af2","#ff6ac1","#ffffff","#a2845e"];
const $ = id => document.getElementById(id);
const esc = s => String(s).replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
let pid = localStorage.getItem("gp_pid");
let couleur = COULEURS[Math.floor(Math.random()*COULEURS.length)];
let decalage = 0, etat = null, secteurAffiche = 0, envoi = false;

function ecran(id){document.querySelectorAll(".ecran").forEach(e=>e.classList.toggle("on", e.id===id));}
function fmt(s){s=Math.max(0,s);const m=Math.floor(s/60), r=s-m*60;return String(m).padStart(2,"0")+":"+r.toFixed(1).padStart(4,"0");}
async function api(url, corps){
  const r = await fetch(url, corps ? {method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify(corps)} : {});
  return r.json();
}

COULEURS.forEach(c=>{const b=document.createElement("button");b.style.background=c;b.onclick=()=>{couleur=c;majCouleurs();};$("couleurs").appendChild(b);});
function majCouleurs(){[...$("couleurs").children].forEach((b,i)=>b.classList.toggle("choisi", COULEURS[i]===couleur));}
majCouleurs();
for(let i=0;i<5;i++){const f=document.createElement("div");f.className="feu";$("g-feux").appendChild(f);}

$("rejoindre").onclick = async () => {
  const nom = $("nom").value.trim();
  if(!nom){$("nom").focus();return;}
  $("rejoindre").disabled = true;
  const r = await api("/api/rejoindre",{nom, couleur});
  $("rejoindre").disabled = false;
  if(r.erreur){ $("j-err").textContent = r.erreur; $("nom").focus(); return; }
  $("j-err").textContent = "";
  pid = r.pid; localStorage.setItem("gp_pid", pid);
  charger();
};
$("nom").addEventListener("keydown", e=>{if(e.key==="Enter")$("rejoindre").click();});

function rendre(d){
  etat = d;
  $("rouge").classList.toggle("on", !!d.pause && !!pid && !d.inconnu);
  if(!pid || d.inconnu){
    if(pid && d.inconnu) $("j-err").textContent = "Vous avez été retiré de la course par la direction de course. Vous pouvez rejoindre la grille à nouveau.";
    localStorage.removeItem("gp_pid"); pid=null; secteurAffiche=0; ecran("e-inscription"); return;
  }
  const m = d.moi;
  ["g-num","c-num"].forEach(id=>{$(id).textContent=m.numero;$(id).style.background=m.couleur;});
  $("g-nom").textContent = m.nom; $("c-nom").textContent = m.nom;
  if(m.fini){ ecran("e-fin"); $("f-temps").textContent = fmt(m.total); $("f-pos").textContent = "P"+m.position+" / "+m.nb_pilotes; return; }
  if(d.secteur){
    ecran("e-course");
    $("c-pos").textContent = "P"+m.position+" / "+m.nb_pilotes;
    $("c-pen").textContent = m.penalite+" s";
    $("c-secs").innerHTML = Array.from({length:d.nb_secteurs},(_,k)=>`<div class="sec ${k<m.secteurs?(m.violets.includes(k)?"violet":"ok"):k===m.secteurs?"ici":""}"></div>`).join("");
    if(d.secteur.numero !== secteurAffiche) afficherSecteur(d.secteur);
    if(d.secteur.indice){$("s-indice").innerHTML="🔧 "+d.secteur.indice;$("s-indice").classList.add("on");}
    return;
  }
  ecran("e-grille");
  $("g-msg").textContent = d.etat==="fini" ? "Course terminée. Regardez le podium au tableau !" : d.etat==="course" ? "Feux rouges… préparez-vous !" : "Vous êtes sur la grille. En attente du départ…";
}

function afficherSecteur(s){
  secteurAffiche = s.numero;
  $("s-num").textContent = "Secteur "+s.numero+" / "+etat.nb_secteurs;
  $("s-titre").textContent = s.titre;
  $("s-notion").textContent = s.notion;
  $("s-enonce").innerHTML = s.enonce;
  $("s-indice").classList.remove("on");
  $("s-retour").className = "retour";
  const pit = '<button class="btn gris" id="pit">🔧 Arrêt au stand (+30 s)</button>';
  if(s.type==="qcm"){
    $("s-saisie").innerHTML = '<div class="qcm">'+s.options.map(o=>`<button data-o="${esc(o)}">${esc(o)}</button>`).join("")+'</div><div class="actions">'+pit+'</div>';
    document.querySelectorAll(".qcm button").forEach(b=>b.onclick=()=>valider(b.dataset.o));
  } else {
    $("s-saisie").innerHTML = '<input type="text" id="rep" placeholder="Votre réponse" autocomplete="off"><div class="actions"><button class="btn" id="valider">Valider</button>'+pit+'</div>';
    $("valider").onclick = () => valider($("rep").value);
    $("rep").addEventListener("keydown", e=>{if(e.key==="Enter")valider($("rep").value);});
    setTimeout(()=>$("rep").focus(),50);
  }
  $("pit").onclick = async () => { const r = await api("/api/stand",{pid}); if(r.indice){$("s-indice").innerHTML="🔧 "+r.indice;$("s-indice").classList.add("on");} charger(); };
  garage(s.outil);
}

async function valider(reponse){
  if(envoi || !String(reponse).trim()) return;
  envoi = true;
  const r = await api("/api/repondre",{pid, reponse});
  envoi = false;
  const z = $("s-retour");
  if(r.ok){
    z.className = "retour on "+(r.violet?"violet":"ok");
    z.innerHTML = r.violet ? "🟣 SECTEUR VIOLET ! Vous êtes le premier à le boucler." : "✅ "+esc(r.message);
    setTimeout(charger, 1100);
  } else {
    z.className = "retour on ko";
    z.innerHTML = "❌ "+esc(r.message)+(r.retour?"<br>"+esc(r.retour):"");
    charger();
  }
}

const OUTILS = {
  decoder: {titre:"Décodeur d'IDs", aide:"Collez une liste d'IDs (séparés par des virgules) : le tokenizer les retransforme en texte.", place:"ex. 23, 456, 789", bouton:"Décoder"},
  tokens: {titre:"Tokenizer", aide:"Tapez un texte : le tokenizer le découpe en tokens et donne l'ID de chacun.", place:"ex. voiture de course", bouton:"Tokeniser"},
  carte: {titre:"Carte de cross-attention", aide:"Ligne = mot écrit par le decoder · colonne = mot français · case = part de l'attention.", place:"une phrase en français", bouton:"Afficher la carte"},
  pas: {titre:"Moteur pas à pas", aide:"L'encoder lit la phrase une fois, puis le decoder écrit token par token. Chaque ligne = un tour du decoder (3 meilleurs choix).", place:"une phrase en français", bouton:"Lancer le moteur"},
  traduire: {titre:"Traducteur", aide:"Testez une phrase et une stratégie de décodage.", place:"une phrase en français", bouton:"Traduire"}
};
const NOTES = {};

function garage(outil){
  if(outil==="cerveau"){
    $("garage").innerHTML = '<h3>🧠 Secteur cerveau</h3><p>Pas d\'outil ici : le modèle ne peut pas répondre à votre place. Raisonnez avec le cours, et posez vos calculs dans le bloc-notes.</p><textarea class="notes" id="notes" placeholder="Vos calculs…"></textarea>';
    const k = secteurAffiche; $("notes").value = NOTES[k]||""; $("notes").oninput = () => NOTES[k] = $("notes").value;
    return;
  }
  const o = OUTILS[outil];
  let extra = "";
  if(outil==="traduire") extra = '<select id="g-strat" style="width:auto"><option>Beam search</option><option>Greedy</option><option>Sampling</option></select><label style="margin:0;display:flex;align-items:center;gap:8px">T <input type="range" id="g-temp" min="0.1" max="2" step="0.1" value="1"><span id="g-tv">1.0</span></label>';
  $("garage").innerHTML = `<h3>${o.titre}</h3><p>${o.aide}</p><div class="ligne-outil"><input type="text" id="g-in" placeholder="${o.place}">${extra}<button class="btn or" id="g-go">${o.bouton}</button></div><div class="res" id="g-res"></div>`;
  if(outil==="traduire") $("g-temp").oninput = () => $("g-tv").textContent = (+$("g-temp").value).toFixed(1);
  const lancer = async () => {
    const texte = $("g-in").value.trim(); if(!texte) return;
    $("g-go").disabled = true; $("g-res").innerHTML = '<p>Le moteur tourne…</p>';
    const corps = {pid, outil, texte};
    if(outil==="traduire"){corps.strategie=$("g-strat").value; corps.temperature=+$("g-temp").value;}
    const r = await api("/api/outil", corps);
    $("g-go").disabled = false;
    $("g-res").innerHTML = r.erreur ? `<p style="color:#ff8a8a">${esc(r.erreur)}</p>` : rendreOutil(outil, r);
  };
  $("g-go").onclick = lancer;
  $("g-in").addEventListener("keydown", e=>{if(e.key==="Enter")lancer();});
}

function couleurPoids(w){return `rgba(255,${Math.round(200-140*w)},${Math.round(60-40*w)},${(0.12+0.88*w).toFixed(2)})`;}

function rendreOutil(outil, r){
  if(outil==="decoder") return `<div class="trad">${esc(r.texte)||"<i>(vide)</i>"}</div>`;
  if(outil==="tokens") return r.tokens.map((t,i)=>`<span class="jeton">${esc(t)}<small>${r.ids[i]}</small></span>`).join("") + `<p style="margin-top:8px">${r.tokens.length} tokens (dont &lt;/s&gt;)</p>`;
  if(outil==="traduire") return `<div class="trad">${esc(r.traduction)}</div>`;
  if(outil==="pas") return `<table class="pas"><tr><th>Tour</th><th>Choisi</th><th>2e</th><th>3e</th></tr>`+r.etapes.map((e,i)=>`<tr><td>${i+1}</td>${e.map(([t,p])=>`<td>${esc(t)} <small>${Math.round(p*100)}%</small></td>`).join("")}</tr>`).join("")+`</table><p style="margin-top:10px">Traduction : <b>${esc(r.traduction)}</b></p>`;
  if(outil==="carte") return `<table class="hm"><tr><th></th>${r.source.map(s=>`<th>${esc(s)}</th>`).join("")}</tr>`+r.cible.map((c,i)=>`<tr><th style="text-align:right">${esc(c)}</th>${r.poids[i].map(w=>`<td style="background:${couleurPoids(w)};color:${w>.45?"#000":"#fff"}">${Math.round(w*100)}%</td>`).join("")}</tr>`).join("")+`</table>`;
  return "";
}

function tick(){
  if(etat){
    const now = Date.now()/1000 + decalage;
    let c = 0;
    if(etat.debut && now>=etat.debut && etat.etat!=="attente") c = (etat.fin || etat.pause || now) - etat.debut - (etat.cumul_pause||0);
    $("c-chrono").textContent = fmt(c);
    const reste = etat.debut ? etat.debut-now : 99;
    const allumes = reste>0 ? Math.min(5, Math.ceil(6-reste)) : 0;
    document.querySelectorAll("#g-feux .feu").forEach((f,i)=>f.classList.toggle("on", etat.etat==="course" && reste>0 && i<allumes));
    $("go").textContent = etat.etat==="course" && reste<=0 && reste>-2 ? "GO !" : "";
  }
  requestAnimationFrame(tick);
}
requestAnimationFrame(tick);

async function charger(){
  try{
    const d = await api("/api/etat?pid="+(pid||""));
    decalage = d.maintenant - Date.now()/1000;
    rendre(d);
  }catch(e){}
}
async function boucle(){ await charger(); setTimeout(boucle, etat && etat.etat==="course" && etat.debut-(Date.now()/1000+decalage) > -3 ? 400 : 1500); }
boucle();
</script>
</body>
</html>
'''
print("Pages prêtes")

### 5. Le serveur

Les adresses du site : `/dashboard`, `/play`, et les routes `/api/...` que les pages appellent chaque seconde.

In [ ]:
import io, re, logging
import segno
from flask import Flask, request, jsonify, Response, redirect

app = Flask(__name__)
logging.getLogger("werkzeug").setLevel(logging.ERROR)


@app.get("/")
def accueil():
    return redirect("/play")


@app.get("/play")
def page_pilote():
    return Response(PAGE_PILOTE, mimetype="text/html")


@app.get("/dashboard")
def page_dashboard():
    return Response(PAGE_DASHBOARD, mimetype="text/html")


@app.get("/favicon.ico")
def favicon():
    return Response(status=204)


@app.get("/qr.svg")
def qr():
    buffer = io.BytesIO()
    segno.make(request.args.get("u", "")[:300], error="m").save(buffer, kind="svg", scale=10, border=2, dark="#07090c", light="#ffffff", omitsize=True, xmldecl=False)
    return Response(buffer.getvalue(), mimetype="image/svg+xml")


@app.get("/api/tableau")
def api_tableau():
    return jsonify(tableau())


@app.get("/api/etat")
def api_etat():
    return jsonify(etat_pilote(request.args.get("pid", "")))


@app.post("/api/rejoindre")
def api_rejoindre():
    d = request.get_json(force=True)
    pid, erreur = rejoindre(d.get("nom"), d.get("couleur"))
    return jsonify({"pid": pid, "erreur": erreur})


@app.post("/api/repondre")
def api_repondre():
    d = request.get_json(force=True)
    if d.get("pid") not in pilotes:
        return jsonify({"ok": False, "message": "Pilote inconnu."})
    return jsonify(repondre(d["pid"], d.get("reponse", "")))


@app.post("/api/stand")
def api_stand():
    d = request.get_json(force=True)
    if d.get("pid") not in pilotes:
        return jsonify({"indice": ""})
    return jsonify(stand(d["pid"]))


@app.post("/api/outil")
def api_outil():
    d = request.get_json(force=True)
    texte = str(d.get("texte", "")).strip()[:200]
    outil = d.get("outil")
    if not texte:
        return jsonify({"erreur": "Écrivez quelque chose."})
    if course["pause"] is not None:
        return jsonify({"erreur": "🚩 Drapeau rouge : le garage est fermé pendant la suspension."})
    try:
        if outil == "decoder":
            ids = [int(x) for x in re.findall(r"\d+", texte) if int(x) < len(tokenizer)]
            return jsonify({"texte": decoder_ids(ids)})
        if outil == "tokens":
            return jsonify(tokens(texte))
        if outil == "carte":
            return jsonify(carte(texte))
        if outil == "pas":
            return jsonify(pas_a_pas(texte))
        if outil == "traduire":
            t = min(2.0, max(0.1, float(d.get("temperature", 1.0))))
            return jsonify({"traduction": traduire(texte, d.get("strategie", "Beam search"), t)})
    except Exception as e:
        return jsonify({"erreur": f"Le moteur a calé : {e}"})
    return jsonify({"erreur": "Outil inconnu."})


@app.post("/api/admin")
def api_admin():
    d = request.get_json(force=True)
    if str(d.get("code")) != code_admin:
        return jsonify({"ok": False}), 403
    admin(d.get("action"), d.get("numero"))
    return jsonify({"ok": True})

### 6. Démarrer le serveur

À exécuter **une seule fois**.

In [ ]:
import threading

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=8765, threaded=True), daemon=True).start()
print("Serveur démarré")

### 7. Ouvrir le lien public 🔗

Cette cellule crée le lien public et affiche :
- le lien du **dashboard** à ouvrir sur le PC branché au projecteur (il contient le code animateur : ne le montrez pas),
- le lien **pilotes** (il est aussi affiché en grand au tableau, avec un QR code).

Le lien peut mettre **10 à 20 secondes** à répondre après son affichage. S'il ne marche pas, relancez seulement cette cellule.

In [ ]:
import subprocess, re, time
from IPython.display import HTML, display

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8765", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
depart = time.time()
while url is None and time.time() - depart < 90:
    trouve = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
    if trouve:
        url = trouve.group(0)
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

if url:
    display(HTML(f'''
    <div style="background:#07090c;color:#fff;font-family:sans-serif;padding:20px;border-radius:12px;border-left:6px solid #ff2d2d">
      <div style="font-size:22px;font-weight:bold">🏁 Le Grand Prix est en ligne</div><br>
      🖥️ <b>Projecteur (animateur)</b> : <a style="color:#ffc83d" href="{url}/dashboard?admin={code_admin}" target="_blank">{url}/dashboard?admin={code_admin}</a><br><br>
      🏎️ <b>Pilotes</b> : <a style="color:#ffc83d" href="{url}/play" target="_blank">{url}/play</a><br><br>
      🔑 Code animateur : <b>{code_admin}</b>
    </div>'''))
else:
    print("Pas de lien public. Relancez cette cellule.")

### 🎬 Déroulé de la course

1. Ouvrez le lien **projecteur** sur le PC branché au vidéoprojecteur, en plein écran (F11). L'écran « Rejoignez la grille » affiche le lien et le QR code.
2. Les étudiants ouvrent le lien sur leur PC, choisissent un pseudo et une couleur : leur voiture apparaît sur la grille.
3. Cliquez sur **Départ** : les 5 feux rouges s'allument (avec le son), s'éteignent… GO !
4. Chaque étudiant boucle les 6 secteurs. Au tableau : les voitures avancent sur le circuit, le premier à boucler un secteur le passe en **violet**, la radio annonce chaque événement.
5. Quand tout le monde a franchi la ligne (ou quand vous cliquez sur **Fin de course**), le **podium** s'affiche.

| Bouton | Effet |
|---|---|
| Départ | Lance (ou relance) la course, remet les chronos à zéro, garde les pilotes |
| Pause / Reprendre | 🚩 Drapeau rouge : le chrono s'arrête pour tout le monde, les réponses et le garage sont bloqués. Reprendre relance le chrono là où il s'était arrêté |
| ✕ à côté d'un nom | Retire ce pilote (écran de grille ou classement). Son écran revient à l'inscription |
| Fin de course | Arrête le chrono et affiche le podium |
| Reset | Retour à l'écran de grille, pilotes conservés |
| Vider la grille | Supprime tous les pilotes |

**Règlement :** mauvaise réponse **+15 s**, arrêt au stand (indice) **+30 s**. Classement : le plus de secteurs, puis le meilleur temps pénalités comprises.

⚠️ Gardez cet onglet Colab ouvert pendant toute la course : s'il se ferme, le serveur s'arrête.

### 🔑 Clé des réponses (animateur)

In [ ]:
#@title Afficher les réponses
for i, r in enumerate([reponse_compte_tours, reponse_masque, "chat,4", "7  (exp(2) ≈ 7,39)", "A cat,0.36", f"{phrase_drapeau}  → cible : {cible_drapeau}"], 1):
    print(f"Secteur {i} : {r}")